In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import clickhouse_connect
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format)
print("✅ Imports OK")

✅ Imports OK


In [2]:
client = clickhouse_connect.get_client(
    host='clickhouse', port=8123,
    user='default', password='clickhouse123',
    database='gold'
)

df = client.query_df("""
SELECT
    d.Year                              AS year,
    d.Month                             AS month,
    d.Quarter                           AS quarter,
    SUM(f.LinePrice)                    AS total_sales,
    SUM(f.Quantity)                     AS total_quantity,
    COUNT(DISTINCT f.DocumentCode)      AS nb_documents,
    COUNT(DISTINCT f.ClientSK)          AS nb_clients,
    COUNT(DISTINCT f.ItemSK)            AS nb_produits,
    COUNT(DISTINCT f.GeographicalAreaSK) AS nb_regions,
    COUNT(DISTINCT f.WarehouseSK)       AS nb_warehouses,
    AVG(f.DiscountPercentage)           AS avg_discount,
    SUM(f.DiscountAmount)               AS total_discount,
    SUM(f.CostPrice)                    AS total_cost,
    SUM(f.LinePrice - f.CostPrice)      AS total_marge
FROM gold.fact_sales AS f
JOIN gold.dim_date AS d ON f.DateSK = d.DateSK
WHERE f.DocumentTypeSK IN (4, 1, 5)
  AND f.LinePrice IS NOT NULL
GROUP BY year, month, quarter
ORDER BY year, month
""")

df['date'] = pd.to_datetime(df[['year', 'month']].assign(day=1))
print(f"✅ {len(df)} mois chargés")
df.head()

✅ 51 mois chargés


,year,month,quarter,total_sales,total_quantity,nb_documents,nb_clients,nb_produits,nb_regions,nb_warehouses,avg_discount,total_discount,total_cost,total_marge,date
0,2021,3,1,404.06,6,1,1,6,1,2,15.00,60.61,334.97,69.09,2021-03-01
1,2022,1,1,"591,407.94",10983,1125,197,3225,9,3,9.28,"44,802.24","604,390.30","-12,982.36",2022-01-01
2,2022,2,1,"576,097.45",10770,1263,190,3327,9,3,9.69,"36,225.76","677,038.55","-100,941.11",2022-02-01
3,2022,3,1,"680,213.89",11488,1304,200,3497,9,3,7.93,"40,184.68","683,998.33","-3,784.44",2022-03-01
4,2022,4,2,"650,789.55",11385,1093,224,3264,9,4,9.09,"47,939.71","638,230.20","12,559.36",2022-04-01


In [3]:
print(f"Avant nettoyage : {len(df)} mois")
print(f"Mois incomplets :")
print(df[
    (df['date'] < '2022-01-01') | (df['date'] >= '2026-02-01')
][['date', 'total_sales']].to_string(index=False))

df = df[
    (df['date'] >= '2022-01-01') & (df['date'] < '2026-02-01')
].reset_index(drop=True)

print(f"\nAprès nettoyage : {len(df)} mois")
print(f"Période : {df['date'].min().strftime('%b %Y')} → {df['date'].max().strftime('%b %Y')}")

Avant nettoyage : 51 mois
Mois incomplets :
      date  total_sales
2021-03-01       404.06
2026-02-01   114,247.11

Après nettoyage : 49 mois
Période : Jan 2022 → Jan 2026


In [5]:
# Variables directement disponibles sans feature engineering complexe
# On garde ce qui est métier-pertinent et statistiquement corrélé à total_sales

df_ml = df[[
    'date',
    'year',
    'month',
    'quarter',
    'total_sales',      # ← TARGET
    'total_quantity',   # ← corrélé au CA
    'nb_documents',     # ← volume d'activité
    'nb_clients',       # ← nombre de clients actifs
]].copy()

# Variables temporelles simples
df_ml['semester'] = df_ml['month'].apply(lambda x: 1 if x <= 6 else 2)
df_ml['is_summer']   = df_ml['month'].isin([7, 8]).astype(int)
df_ml['is_end_year'] = df_ml['month'].isin([11, 12]).astype(int)
df_ml['is_january']  = (df_ml['month'] == 1).astype(int)

# Lag features — uniquement les plus pertinents
df_ml['lag_1']  = df_ml['total_sales'].shift(1)
df_ml['lag_12'] = df_ml['total_sales'].shift(12)  # même mois année précédente
df_ml['rolling_mean_3'] = df_ml['total_sales'].shift(1).rolling(3).mean()

# Supprimer les NaN dus aux lags
df_ml = df_ml.dropna().reset_index(drop=True)

print(f"✅ Dataset prêt : {df_ml.shape}")
print(f"Période : {df_ml['date'].min().strftime('%b %Y')} → {df_ml['date'].max().strftime('%b %Y')}")
df_ml.head()

✅ Dataset prêt : (37, 15)
Période : Jan 2023 → Jan 2026


,date,year,month,quarter,total_sales,total_quantity,nb_documents,nb_clients,semester,is_summer,is_end_year,is_january,lag_1,lag_12,rolling_mean_3
0,2023-01-01,2023,1,1,"1,037,923.46",16598,1380,223,1,0,0,1,"985,659.74","591,407.94","960,005.64"
1,2023-02-01,2023,2,1,"939,044.09",15537,1280,227,1,0,0,0,"1,037,923.46","576,097.45","993,566.44"
2,2023-03-01,2023,3,1,"948,325.78",15961,1392,240,1,0,0,0,"939,044.09","680,213.89","987,542.43"
3,2023-04-01,2023,4,2,"913,161.55",17831,1237,235,1,0,0,0,"948,325.78","650,789.55","975,097.78"
4,2023-05-01,2023,5,2,"1,184,666.29",20290,1514,246,1,0,0,0,"913,161.55","734,335.68","933,510.47"


In [7]:
from sklearn.feature_selection import RFE, SequentialFeatureSelector
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, r2_score
import pandas as pd
import numpy as np

# Toutes les features candidates
ALL_FEATURES = [
    'month', 'quarter', 'semester',
    'total_quantity', 'nb_documents', 'nb_clients',
    'nb_produits', 'nb_regions', 'avg_discount',
    'total_discount', 'total_cost', 'total_marge',
    'is_summer', 'is_end_year', 'is_january',
    'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12',
    'rolling_mean_3', 'rolling_mean_6', 'rolling_std_3',
    'rolling_max_3', 'rolling_min_3',
    'growth_1m', 'growth_yoy'
]
TARGET = 'total_sales'
# Garder uniquement les colonnes existantes dans df_ml
ALL_FEATURES = [f for f in ALL_FEATURES if f in df_ml.columns]

X_all = df_ml[ALL_FEATURES]
y_all = df_ml[TARGET]

# Split temporel
split_idx    = int(len(df_ml) * 0.9)
X_tr, X_te   = X_all.iloc[:split_idx], X_all.iloc[split_idx:]
y_tr, y_te   = y_all.iloc[:split_idx], y_all.iloc[split_idx:]

scaler    = StandardScaler()
X_tr_sc   = scaler.fit_transform(X_tr)
X_te_sc   = scaler.transform(X_te)

print(f"✅ {len(ALL_FEATURES)} features candidates")
print(ALL_FEATURES)

✅ 12 features candidates
['month', 'quarter', 'semester', 'total_quantity', 'nb_documents', 'nb_clients', 'is_summer', 'is_end_year', 'is_january', 'lag_1', 'lag_12', 'rolling_mean_3']


In [8]:
# ─── 1. RFE — Recursive Feature Elimination ───────────────────
print("\n" + "="*55)
print("1️⃣  RFE — Recursive Feature Elimination")
print("="*55)

rfe = RFE(estimator=LinearRegression(), n_features_to_select=8)
rfe.fit(X_tr_sc, y_tr)

features_rfe = [ALL_FEATURES[i] for i in range(len(ALL_FEATURES)) if rfe.support_[i]]
pred_rfe     = rfe.predict(X_te_sc)
mape_rfe     = np.mean(np.abs((y_te - pred_rfe) / y_te)) * 100
r2_rfe       = r2_score(y_te, pred_rfe)
mae_rfe      = mean_absolute_error(y_te, pred_rfe)

print(f"Features sélectionnées ({len(features_rfe)}) : {features_rfe}")
print(f"MAE  : {mae_rfe:>12,.0f} DT")
print(f"MAPE : {mape_rfe:>12.2f} %")
print(f"R²   : {r2_rfe:>12.4f}")



1️⃣  RFE — Recursive Feature Elimination
Features sélectionnées (8) : ['month', 'quarter', 'semester', 'total_quantity', 'nb_documents', 'is_summer', 'lag_12', 'rolling_mean_3']
MAE  :       56,444 DT
MAPE :         4.22 %
R²   :       0.8131


In [9]:
# ─── 2. Forward Selection ──────────────────────────────────────
print("\n" + "="*55)
print("2️⃣  Forward Selection")
print("="*55)

sfs_forward = SequentialFeatureSelector(
    LinearRegression(),
    n_features_to_select=8,
    direction='forward',
    scoring='neg_mean_absolute_error',
    cv=TimeSeriesSplit(n_splits=3)
)
sfs_forward.fit(X_tr_sc, y_tr)

features_fwd = [ALL_FEATURES[i] for i in range(len(ALL_FEATURES)) if sfs_forward.get_support()[i]]
X_te_fwd     = scaler.transform(X_te[ALL_FEATURES])[: , sfs_forward.get_support()]
pred_fwd     = LinearRegression().fit(
                   X_tr_sc[:, sfs_forward.get_support()], y_tr
               ).predict(X_te_fwd)

mape_fwd = np.mean(np.abs((y_te - pred_fwd) / y_te)) * 100
r2_fwd   = r2_score(y_te, pred_fwd)
mae_fwd  = mean_absolute_error(y_te, pred_fwd)

print(f"Features sélectionnées ({len(features_fwd)}) : {features_fwd}")
print(f"MAE  : {mae_fwd:>12,.0f} DT")
print(f"MAPE : {mape_fwd:>12.2f} %")
print(f"R²   : {r2_fwd:>12.4f}")


2️⃣  Forward Selection
Features sélectionnées (8) : ['quarter', 'semester', 'total_quantity', 'nb_documents', 'is_end_year', 'is_january', 'lag_12', 'rolling_mean_3']
MAE  :       61,415 DT
MAPE :         4.54 %
R²   :       0.8031


In [10]:
# ─── 3. Backward Elimination ──────────────────────────────────
print("\n" + "="*55)
print("3️⃣  Backward Elimination")
print("="*55)

sfs_backward = SequentialFeatureSelector(
    LinearRegression(),
    n_features_to_select=8,
    direction='backward',
    scoring='neg_mean_absolute_error',
    cv=TimeSeriesSplit(n_splits=3)
)
sfs_backward.fit(X_tr_sc, y_tr)

features_bwd = [ALL_FEATURES[i] for i in range(len(ALL_FEATURES)) if sfs_backward.get_support()[i]]
X_te_bwd     = scaler.transform(X_te[ALL_FEATURES])[:, sfs_backward.get_support()]
pred_bwd     = LinearRegression().fit(
                   X_tr_sc[:, sfs_backward.get_support()], y_tr
               ).predict(X_te_bwd)

mape_bwd = np.mean(np.abs((y_te - pred_bwd) / y_te)) * 100
r2_bwd   = r2_score(y_te, pred_bwd)
mae_bwd  = mean_absolute_error(y_te, pred_bwd)

print(f"Features sélectionnées ({len(features_bwd)}) : {features_bwd}")
print(f"MAE  : {mae_bwd:>12,.0f} DT")
print(f"MAPE : {mape_bwd:>12.2f} %")
print(f"R²   : {r2_bwd:>12.4f}")


3️⃣  Backward Elimination
Features sélectionnées (8) : ['month', 'quarter', 'total_quantity', 'nb_documents', 'is_end_year', 'lag_1', 'lag_12', 'rolling_mean_3']
MAE  :       69,342 DT
MAPE :         5.12 %
R²   :       0.7634
